# Rule-level confidence intervals — canonical CSV for the article figures

Computes per-rule uplift / realistic-gain CIs for each of the four article datasets with all five CI methods, and saves a tidy CSV plus bootstrap sample arrays.

**Outputs:**
- `article/results/rule_level_cis.csv` — one row per (dataset × rule × method)
- `article/results/rule_level_bootstrap_samples.npz` — bootstrap samples for a single rule (Employee Attrition rule 0) used by Figure 1

These artefacts power Figures 1-4 and Table 1 (rule counts).


In [1]:
from __future__ import annotations
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
REPO = Path.cwd()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO / 'notebooks' / 'article') not in sys.path:
    sys.path.insert(0, str(REPO / 'notebooks' / 'article'))
warnings.filterwarnings('ignore')
from _datasets import load_all
from action_rules import ActionRules
OUT_CSV = REPO / 'article' / 'results' / 'rule_level_cis.csv'
OUT_NPZ = REPO / 'article' / 'results' / 'rule_level_bootstrap_samples.npz'
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [2]:
N_BOOTSTRAP = 500
N_MC = 10000
SEED = 42
METHOD_SPECS = [
    ('bootstrap_percentile', dict(method='bootstrap', bootstrap_type='percentile')),
    ('bootstrap_bca',        dict(method='bootstrap', bootstrap_type='bca')),
    ('wald',                 dict(method='analytic', analytic_type='wald')),
    ('wilson',               dict(method='analytic', analytic_type='wilson')),
    ('bayesian',             dict(method='bayesian')),
]

In [3]:
rows = []
saved_samples = {}
for cfg in load_all():
    print(f'=== {cfg.name} ===')
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    ar.fit(cfg.df, stable_attributes=cfg.stable_attributes, flexible_attributes=cfg.flexible_attributes,
           target=cfg.target, target_undesired_state=cfg.target_undesired_state,
           target_desired_state=cfg.target_desired_state, use_sparse_matrix=cfg.use_sparse_matrix)
    print(f'  {len(ar.output.action_rules)} rules')
    for label, kw in METHOD_SPECS:
        call_kw = dict(kw, confidence_level=0.95, random_state=SEED)
        if kw['method'] == 'bootstrap':
            call_kw['n_bootstrap'] = N_BOOTSTRAP
        elif kw['method'] == 'bayesian':
            call_kw['n_mc'] = N_MC
        results = ar.confidence_intervals(cfg.df, **call_kw)
        for r in results:
            rows.append({
                'dataset': cfg.name,
                'rule_index': r.rule_index,
                'method': label,
                'support': r.support,
                'confidence': r.confidence,
                'uplift_point': r.uplift_point,
                'uplift_ci_lower': r.uplift_ci_lower,
                'uplift_ci_upper': r.uplift_ci_upper,
                'uplift_se': r.uplift_se,
                'gain_point': r.realistic_rule_gain_point,
                'gain_ci_lower': r.realistic_rule_gain_ci_lower,
                'gain_ci_upper': r.realistic_rule_gain_ci_upper,
                'gain_se': r.realistic_rule_gain_se,
            })
        # Save bootstrap samples for fig1 (one rule of Employee Attrition).
        if label == 'bootstrap_percentile' and cfg.name == 'IBM Employee Attrition':
            # Pick the rule whose CI is widest -- most interesting bootstrap distribution.
            widths = [(r.uplift_ci_upper - r.uplift_ci_lower, i, r) for i, r in enumerate(results)]
            widths.sort(reverse=True)
            chosen = widths[0][2] if widths else None
            if chosen is not None and chosen.samples_uplift is not None:
                saved_samples['attrition_rule_uplift'] = chosen.samples_uplift
                saved_samples['attrition_rule_gain'] = chosen.samples_gain if chosen.samples_gain is not None else np.zeros(0)
                saved_samples['attrition_rule_point'] = np.array([chosen.uplift_point])
                saved_samples['attrition_rule_ci'] = np.array([chosen.uplift_ci_lower, chosen.uplift_ci_upper])
                saved_samples['attrition_rule_index'] = np.array([chosen.rule_index])
                # Boolean masks for the chosen rule -- enable Fig 1 notebook to
                # replicate the engine's nonparametric bootstrap and to compute
                # component-level p_u / p_d distributions for the article's
                # composition figure.  uint8 to keep the NPZ compact.
                from action_rules.inference.base import extract_rule_masks
                from action_rules.inference.bootstrap import _precompute_rule_masks
                rule_masks_list = extract_rule_masks(ar.output)
                rm = rule_masks_list[chosen.rule_index]
                u_ante, u_match, d_ante, d_match = _precompute_rule_masks(cfg.df, rm)
                saved_samples['attrition_rule_u_ante'] = u_ante.astype(np.uint8)
                saved_samples['attrition_rule_u_match'] = u_match.astype(np.uint8)
                saved_samples['attrition_rule_d_ante'] = d_ante.astype(np.uint8)
                saved_samples['attrition_rule_d_match'] = d_match.astype(np.uint8)
                saved_samples['attrition_rule_N'] = np.array([len(cfg.df)])
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
np.savez_compressed(OUT_NPZ, **saved_samples)
print(f'\nwrote {OUT_CSV.relative_to(REPO)} ({df.shape[0]:,} rows)')
print(f'wrote {OUT_NPZ.relative_to(REPO)}')
df.head()

=== Telco Customer Churn ===


  24 rules


=== UCI Bank Marketing ===


  24 rules


=== IBM Employee Attrition ===


  21 rules


=== Taiwan Credit Card Default ===


  20 rules



wrote article\results\rule_level_cis.csv (445 rows)
wrote article\results\rule_level_bootstrap_samples.npz


,dataset,rule_index,method,support,confidence,uplift_point,uplift_ci_lower,uplift_ci_upper,uplift_se,gain_point,gain_ci_lower,gain_ci_upper,gain_se
0,Telco Customer Churn,0,bootstrap_percentile,439,0.635535,0.016575,0.011991,0.021300,0.002366,106.303384,78.673455,133.910803,14.459165
1,Telco Customer Churn,1,bootstrap_percentile,439,0.635535,0.026999,0.022073,0.032536,0.002887,157.772069,125.459210,187.677355,15.938543
2,Telco Customer Churn,2,bootstrap_percentile,439,0.635535,0.033223,0.028583,0.037836,0.002429,212.987360,190.233720,235.761450,11.795556
3,Telco Customer Churn,3,bootstrap_percentile,552,0.612319,0.019131,0.012924,0.024801,0.002974,97.532470,67.526434,125.418132,14.882781
4,Telco Customer Churn,4,bootstrap_percentile,552,0.612319,0.038342,0.032593,0.044329,0.003047,162.634794,138.778985,188.378335,12.842122
